# MTPL frequency pricing model

This notebook is the complete analyst workflow: settings, SQL, feature transforms, model definition, fitting, publication, and optional review/deployment. Generated audit identifiers and SQL plumbing stay behind the helper functions.

In [ ]:
# Remote mode loads your private module from PRICING_RUNTIME_MODULE.
DATABASE_MODE = "local"  # "local" or "remote"
RUNTIME_MODULE = None  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False
REFRESH_LOCAL_RAW = False

DATA_AS_OF = "2026-06-30"
SCORING = ("deviance", "nll", "gini")

RUN_EDITOR = False
EDIT_REASON = ""

DEPLOY = False
DEPLOYMENT_REASON = ""

## Analyst settings

Local mode creates and reuses model-local SQLite files. Remote mode loads the private work runtime from `PRICING_RUNTIME_MODULE`, verifies the database name, and refuses writes until explicitly enabled. `DATA_AS_OF` is the source-data cutoff represented by this frame; it is not a deployment date.

In [ ]:
from pathlib import Path
import sys

_search_root = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in (_search_root, *_search_root.parents)
        if (root / "pricing_pipeline").is_dir() and (root / "pricing_models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the pricing repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sqlalchemy import text  # noqa: E402
from superglm import Categorical, Numeric, Spline, SuperGLM  # noqa: E402

from pricing_pipeline.data.fremtpl import load_fremtpl_raw  # noqa: E402
from pricing_pipeline.infra.schema import schema_names_from_connectable  # noqa: E402
from pricing_pipeline.models.config import ValidationSplitConfig  # noqa: E402
from pricing_pipeline.notebook import (  # noqa: E402
    PricingModelSpec,
    build_candidate,
    connect,
    deploy_package,
    open_candidate,
    publish_candidate,
    publish_edits,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/mtpl_frequency"

## Model decisions

Features, validation, offset handling, and model semantics stay visible Python.

In [ ]:
FEATURES = {
    "VehAge": Spline(),
    "DrivAge": Spline(),
    "BonusMalus": Spline(),
    "LogDensity": Numeric(),
    "Area": Categorical(),
    "VehPower": Categorical(),
    "VehBrand": Categorical(),
    "VehGas": Categorical(),
    "Region": Categorical(),
}
superglm_model = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=256,
    features=FEATURES,
)

MODEL = PricingModelSpec(
    name="MTPL_FREQ",
    label="Motor frequency",
    target="ClaimNb",
    model_type="superglm_poisson",
    deployment_slot="MTPL_FREQ_UAT",
    features=tuple(FEATURES),
    dataset_name="freMTPL2freq_model_frame",
    source_system="freMTPL_raw_sql",
    pk_columns=("IDpol",),
    offset_column="LogExposure",
    offset_source_column="Exposure",
    offset_label="log(Exposure)",
    sample_weight_column=None,
    export_weight_column="Exposure",
    validation=ValidationSplitConfig.kfold(
        n_splits=5,
        random_state=42,
        shuffle=True,
    ),
    scoring=SCORING,
)

## Connect, confirm, and register the stable model identity

Read the displayed destination before continuing. Registration is idempotent and SQL owns `model_id`.

In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(
    {
        "Destination": pricing.destination,
        "Artifact root": str(pricing.settings.workbench_artifact_root),
    }
)
model = register_model(
    pricing,
    MODEL,
    source_root=MODEL_DIR,
)
display({"Model": model.name})

## Read source data from SQL

Local mode downloads the full OpenML source when the table is empty and reuses it on later runs. Set `REFRESH_LOCAL_RAW` to replace the local copy. Remote mode never loads OpenML. Keep the primary key and every column needed for transforms, fitting, and export. At work, replace only this cell with the existing private SQL helper when the source database differs from the audit destination.

In [ ]:
if pricing.mode == "local":
    local_source_rows = load_fremtpl_raw(pricing.engine, replace=REFRESH_LOCAL_RAW)
    display({"Local source rows": local_source_rows})

schemas = schema_names_from_connectable(pricing.engine)
SOURCE_SQL = f"""
SELECT
    IDpol, ClaimNb, Exposure, Area, VehPower, VehAge, DrivAge,
    BonusMalus, VehBrand, VehGas, Density, Region
FROM {schemas.pricing}.FREMTPL_RAW
ORDER BY IDpol
"""
raw = pd.read_sql_query(text(SOURCE_SQL), pricing.engine)
if raw.empty:
    raise RuntimeError("FREMTPL_RAW is empty; load source rows before modelling.")
display({"Rows loaded": len(raw), "Columns loaded": len(raw.columns)})

## Build the final model frame

Feature and offset transforms remain ordinary, visible Python. `MODEL` names the fitted offset, its raw export values, and any independent fit/export weights. The helper derives the remaining audit metadata.

In [ ]:
frame = raw.loc[raw["Exposure"].astype(float) > 0].copy()
frame["LogDensity"] = np.log(frame["Density"].astype(float).clip(lower=1.0))
frame["LogExposure"] = np.log(frame["Exposure"].astype(float))
frame = (
    frame.loc[:, ["IDpol", "ClaimNb", "Exposure", "LogExposure", *FEATURES]]
    .sort_values("IDpol")
    .reset_index(drop=True)
)
display({"Rows modelled": len(frame), "Columns modelled": len(frame.columns)})

## Fit and capture audit evidence

The analyst supplies the model frame and its data cutoff. Generated manifests, folds, hashes, versions, and offset-export metadata are plumbing.

In [ ]:
candidate = build_candidate(
    pricing,
    model=model,
    frame=frame,
    superglm_model=superglm_model,
    data_as_of=DATA_AS_OF,
)

## Inspect held-out validation before package publication

In [ ]:
display(candidate.validation_metrics)

## Publish the immutable candidate

This writes the audit lineage and rating tables to SQL. It does not create a deployment or invent an effective date.

In [ ]:
published = publish_candidate(pricing, candidate)
display(
    {
        "Model": published.model_name,
        "Model version": published.model_version,
        "Package": published.package_version,
        "State": published.package_status,
    }
)

## Optional market edit (remote mode only)

Enable `RUN_EDITOR` in the top action cell, make changes in the editor, provide a reason, and publish an immutable child package.

In [ ]:
from superglm.editor import EditorSession

reviewed = None
editor_session = None
editor_widget = None
if RUN_EDITOR:
    reviewed = open_candidate(
        pricing, model=model, package_version=published.package_version
    )
    editor_session = EditorSession.from_model(
        reviewed.bundle.fitted_model,
        train_data=(
            reviewed.bundle.X,
            reviewed.bundle.y,
            reviewed.bundle.sample_weight,
            reviewed.bundle.offset,
        ),
        cv_report=reviewed.bundle.cv_report,
    )
    editor_widget = editor_session.widget()
    display(editor_widget)

## Materialize an in-memory preview of the edits

This ordinary SuperGLM action does not save, publish, deploy, or reopen anything.

In [ ]:
edited_model = None
if RUN_EDITOR:
    edited_model = editor_session.to_model()
edited_model

In [ ]:
edited = None
if RUN_EDITOR:
    if not EDIT_REASON.strip():
        raise ValueError("Describe the market or underwriting reason for the edit.")
    edited = publish_edits(
        pricing,
        candidate=reviewed,
        editor_session=editor_session,
        reason=EDIT_REASON,
    )
    reviewed = open_candidate(
        pricing, model=model, package_version=edited.package_version
    )
    display(
        {
            "Model": model.name,
            "Package": edited.package_version,
            "State": edited.package_status,
        }
    )

## Optional deployment (remote mode only)

Deployment is a separate deliberate action. SQL records its actual activation timestamp.

In [ ]:
if DEPLOY:
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the live package.")
    if reviewed is None:
        reviewed = open_candidate(
            pricing, model=model, package_version=published.package_version
        )
    deployment = deploy_package(
        pricing,
        package=reviewed,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)